# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mirageroy-dev/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

A page is worth reviewing when it is getting meaningful impressions, its position is slipping, and the content is getting old. I will rank pages using a transparent score based on these signals. The reason codes will explain whether a page is old, has high visibility, or is losing position

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

I will score each page using simple, transparent rules based on whether the page is old, has meaningful impressions, and is losing position. Pages with more warning signals will receive higher scores and will be ranked first. The ranked results will be saved in work/outputs/baseline_action_score.csv.

In [11]:
!find /content/flyrank-ml-internship -type f -name "*.csv"


/content/flyrank-ml-internship/outputs/refresh_queue_sample.csv
/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv
/content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


In [12]:
!git clone https://github.com/Mirageroy-dev/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 133 (delta 43), reused 96 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 1.85 MiB | 7.01 MiB/s, done.
Resolving deltas: 100% (43/43), done.


In [13]:
!find /content -type f -name "*.csv"

/content/flyrank-ml-internship/outputs/refresh_queue_sample.csv
/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv
/content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
/content/flyrank-ml-internship/flyrank-ml-internship/outputs/refresh_queue_sample.csv
/content/flyrank-ml-internship/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
/content/sample_data/mnist_test.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/california_housing_test.csv


In [14]:
import os
import pandas as pd

# Go to the cloned repository
%cd /content/flyrank-ml-internship

# Create the processed feature file if needed
if not os.path.exists("data/processed/refresh_feature_vector.csv"):
    !python scripts/01_prepare_features.py

# Load the processed data
df = pd.read_csv("data/processed/refresh_feature_vector.csv")

# Show total rows
print(f"Loaded {len(df)} pages")

# --------------------------------------------------
# CREATE SCORING SIGNALS
# --------------------------------------------------

# 1. A page is stale if it has not been updated recently
df["stale"] = (
    pd.to_numeric(df["days_since_last_update"], errors="coerce") > 180
).astype(int)

# 2. A page has high visibility if it receives many impressions
df["visible"] = (
    pd.to_numeric(df["impressions_90d"], errors="coerce") > 500
).astype(int)

# 3. A page is declining if its trend is going down
if "is_declining_label" in df.columns:
    df["declining"] = (
        df["is_declining_label"]
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes"])
    ).astype(int)
else:
    df["declining"] = (
        df["trend_direction"]
        .astype(str)
        .str.lower()
        .eq("down")
    ).astype(int)

# --------------------------------------------------
# CREATE BASELINE SCORE
# --------------------------------------------------

df["baseline_score"] = (
    df["stale"]
    + df["visible"]
    + df["declining"]
)

# --------------------------------------------------
# CREATE REASON CODES
# --------------------------------------------------

def get_reason_codes(row):
    reasons = []

    if row["stale"] == 1:
        reasons.append("stale_page")

    if row["visible"] == 1:
        reasons.append("high_visibility")

    if row["declining"] == 1:
        reasons.append("position_slipping")

    return "; ".join(reasons) if reasons else "general_review"


df["reason_codes"] = df.apply(get_reason_codes, axis=1)

# --------------------------------------------------
# RANK PAGES
# --------------------------------------------------

queue = df.sort_values(
    by=["baseline_score", "impressions_90d"],
    ascending=[False, False]
).copy()

# Add ranking numbers
queue["baseline_rank"] = range(1, len(queue) + 1)

# --------------------------------------------------
# SAVE OUTPUT
# --------------------------------------------------

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("\nBaseline ranked queue created successfully!")
print(f"Saved {len(queue)} ranked pages to:")
print("work/outputs/baseline_action_score.csv\n")

# Show top 20
display(
    queue[
        [
            "baseline_rank",
            "baseline_score",
            "reason_codes"
        ]
    ].head(20)
)

/content/flyrank-ml-internship
Loaded 30000 pages

Baseline ranked queue created successfully!
Saved 30000 ranked pages to:
work/outputs/baseline_action_score.csv



,baseline_rank,baseline_score,reason_codes
16751,1,3,stale_page; high_visibility; position_slipping
16514,2,3,stale_page; high_visibility; position_slipping
7021,3,3,stale_page; high_visibility; position_slipping
21268,4,3,stale_page; high_visibility; position_slipping
11489,5,3,stale_page; high_visibility; position_slipping
12045,6,3,stale_page; high_visibility; position_slipping
698,7,3,stale_page; high_visibility; position_slipping
5327,8,3,stale_page; high_visibility; position_slipping
26810,9,3,stale_page; high_visibility; position_slipping
20837,10,3,stale_page; high_visibility; position_slipping


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 review

The highest-priority pages should be reviewed first. For ranks 1–16, the recommended action is to review and refresh the content because the pages are stale, have high visibility, and show a declining position trend. The reason code is stale_page; high_visibility; position_slipping. Confidence is high because all three scoring signals are present. This prioritization could be wrong if the decline is temporary or if the page was recently updated but the freshness data is inaccurate.

For ranks 17–20, the recommended action is also to review the content, especially because the pages have high visibility and are showing a declining position trend. The reason code is high_visibility; position_slipping. Confidence is moderate to high because two strong warning signals are present. This prioritization could be wrong if the ranking decline is caused by normal short-term variation or seasonality.

Overall, the baseline score provides a transparent way to prioritize pages. Higher scores indicate that more warning signals are present, so those pages should generally be reviewed sooner.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks + leakage check

The weakest picks are pages with fewer warning signals. A page may receive a lower score because it is not stale, does not have high visibility, or is not showing a declining trend. These pages may still need review, but there is less evidence that they should be prioritized over higher-scoring pages.

The scoring logic only uses the defined page features for staleness, visibility, and trend direction. No product flags, client names, URLs, private queries, or future performance windows were intentionally used in the ranking logic. The approach is based on observed features and is intended as a transparent baseline for prioritization rather than a claim that every high-scoring page definitely needs to be refreshed.

In [16]:
-# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.